In [ ]:
# Sara Vanaki
# 10/9/2026
# script for fine tuning a 12 lead model on 1 lead data using light resnet style architecture
# 12-lead model had val auc and test auc of .93

# setting num threads for libraries to single thread so the CPU isn't context switching over many tasks
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"


# regular import statements for all we need! need to import after ^^ bc system reads env variables at init time
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader
# groupshufflesplit helps us to do splits on patients so no data leakage
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve

# Set process affinity / threads for yoda
os.sched_setaffinity(0, set(range(30)))
torch.set_num_threads(4)

try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

# Step 1) Path configs, global vars, and leads to finetune model on
# -------------------------------------------------------------------------
base_path = '/home/sv67/Donnchadh/datasets/mimic/mimic-iv-ecg/'
h5_path = os.path.join(base_path, 'mimic_iv_ecg_full.h5')
clean_csv_path = os.path.join(base_path, 'mimic_iv_ecg_clean_metadata.csv')
pretrained_12lead_path = os.path.join(base_path, 'saved_models/2026.10.06/best_ecg_sex_12lead_model_val_150k.pth')

output_path = os.path.join(base_path, 'saved_models/2026.10.09/')
# make dir if it DNE
os.makedirs(output_path, exist_ok=True)

BATCH_SIZE = 128
EPOCHS = 15             # 15 epochs bc lower didn't have time to learn
LR = 1e-4
SAMPLES_COUNT = 20_000  # can change if you want, but more has diminishing returns

# List of leads you want to fine-tune on
LEADS_CONFIG =[
    {"tag": "leadv1", "idx": 6, "display" : "Lead V1"},
]

# Step 2) Data Subsampling & Patient-wise Split (SAME ACROSS ALL LEADS)
# -------------------------------------------------------------------------
print("Loading pre-filtered clean metadata...")
df = pd.read_csv(clean_csv_path)

np.random.seed(42)
unique_subjects = df['subject_id'].unique()
sampled_subjects = np.random.choice(unique_subjects, size = min(SAMPLES_COUNT, len(unique_subjects)), replace=False)
df_clean = df[df['subject_id'].isin(sampled_subjects)].reset_index(drop=True)

print(f"Loaded {len(df_clean):,} valid ECGs across {len(sampled_subjects):,} sampled subjects.")

# Patient-wise 3-Way Split (70% Train / 15% Validation / 15% Test)
gss_test = GroupShuffleSplit(n_splits=1, test_size = 0.15, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df_clean, groups=df_clean['subject_id']))

train_val_df = df_clean.iloc[train_val_idx].reset_index(drop=True)
test_df = df_clean.iloc[test_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size = 0.1765, random_state=42)
train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['subject_id']))

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print(f"Train ECGS: {len(train_df):,} | Val ECGS: {len(val_df):,} | Test ECGs: {len(test_df):,}\n")

# Step 3) HDF5 Dataset Class Def
# -------------------------------------------------------------------------
class MIMICECGSingleLeadDataset(Dataset):
    def __init__(self, df, h5_path, lead_idx=1):
        self.df = df
        self.h5_path = h5_path
        self.lead_idx = lead_idx
        self.h5_file = None
    def _open_h5(self):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        self._open_h5()
        row = self.df.iloc[idx]
        try:
            # get the specific index i want 
            h5_idx = int(row['h5_idx'])
            # loads a 3d slice from the h5 file at the sample i want, a single lead ecg signal, and all time points
            sig = self.h5_file['signals'][h5_idx, self.lead_idx: self.lead_idx+1,:]
            # converting all bad nums to 0 so we don't mess up gradients to lead to exploding gradient problem
            sig = np.nan_to_num(sig, nan=0.0, posinf = 0.0, neginf=0.0)
            
            # standardizes the signal using z-score normalization along the time axis w zero division protection w small val
            mean = np.mean(sig, axis=1, keepdims=True)
            std = np.std(sig, axis = 1, keepdims=True)
            std = np.where(std < 1e-6, 1.0, std)
            sig = (sig - mean) / std

            # converts both signal and label into tensor tuple (X,y)
            return torch.tensor(sig, dtype=torch.float32), torch.tensor(row['label'], dtype=torch.float32)
        except Exception:
            return torch.zeros((1, 5000), dtype = torch.float32), torch.tensor(row['label'], dtype=torch.float32)
        
# Step 4) Model Architecture Definition
# i tried deeper layers, aka more resblocks but that didn't seem to help, learning got worse tbh
# -------------------------------------------------------------------------
class ResBlock1D(nn.Module):
    def __init__(self, channels, dropout_rate = 0.2):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size = 7, padding = 3)
        self.bn1 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout1d(dropout_rate)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size = 7, padding = 3)
        self.bn2 = nn.BatchNorm1d(channels)
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out += residual
        return self.relu(out)

class ECGSexClassifier(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size = 15, stride = 2, padding = 7),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size = 3, stride = 2, padding = 1)
        )

        self.layer1 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size = 7, stride = 2, padding = 3),
            nn.BatchNorm1d(128), 
            nn.ReLU()
        )
        self.res1 = ResBlock1D(128)
        self.layer2 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size = 7, stride = 2, padding = 3),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )
        self.res2 = ResBlock1D(256)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.res1(self.layer1(x))
        x = self.res2(self.layer2(x))
        x = self.global_pool(x).squeeze(-1)
        return self.fc(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
all_summary_results = []

# Step 5) Automated pipeline loop over all leads i want
# -------------------------------------------------------------------------
for lead_cfg in LEADS_CONFIG:
    tag = lead_cfg["tag"]
    lead_idx = lead_cfg["idx"]
    display_name = lead_cfg["display"]

    print("\n" + "=" *70)
    print(f" STARTING FINE-TUNING FOR: {display_name.upper()} (Index: {lead_idx})")
    print("="*70)

    best_model_path = os.path.join(output_path, f'2_best_ecg_sex_{tag}_finetuned_15epochs.pth')
    best_outputs_path = os.path.join(output_path, f'2_best_epoch_outputs_{tag}_finetuned_15epochs.npz')

    # DataLoaders for specific lead
    # we use num workers for how many cpu subprocesses work in parallel to load and preprocess
    # pin memory speeds up the transfer of the data from CPU ram to gpu vram
    train_loader = DataLoader(
        MIMICECGSingleLeadDataset(train_df, h5_path, lead_idx),
        batch_size = BATCH_SIZE, shuffle = True, num_workers = 16, pin_memory = True, persistent_workers=True
    )
    val_loader = DataLoader(
        MIMICECGSingleLeadDataset(val_df, h5_path, lead_idx),
        batch_size = BATCH_SIZE, shuffle = False, num_workers = 6, pin_memory = True
    )
    test_loader = DataLoader(
        MIMICECGSingleLeadDataset(test_df, h5_path, lead_idx),
        batch_size = BATCH_SIZE, shuffle = False, num_workers = 6, pin_memory = True
    )

    # Re-instantiate fresh model and load pre-trained weights
    model = ECGSexClassifier(in_channels=1).to(device)
    state_dict = torch.load(pretrained_12lead_path, map_location = device)
    
    # Adapt 12-channel stem to 1 channel
    old_stem_weight = state_dict['stem.0.weight']
    state_dict['stem.0.weight'] = old_stem_weight.mean(dim=1, keepdim = True) * 3.0
    model.load_state_dict(state_dict)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay = 1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = EPOCHS, eta_min = 1e-5)
    # we use a scaler to prevent gradient underflow (similar to vanishing gradient) but bc rounding bits go to 0
    scaler = torch.amp.GradScaler('cuda')

    best_val_auc = 0.0
    best_epoch = 0

    # Training Loop
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for signals, labels in train_loader:
            signals, labels = signals.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                logits = model(signals).squeeze(-1)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * signals.size(0)
        train_loss /= len(train_loader.dataset)

        # Validation Loop
        model.eval()
        val_loss = 0.0
        val_preds, val_labels = [], []
        with torch.no_grad():
            for signals, labels in val_loader:
                signals, labels = signals.to(device), labels.to(device)
                logits = model(signals).squeeze(-1)
                loss = criterion(logits, labels)
                probs = torch.sigmoid(logits)

                val_loss += loss.item() * signals.size(0)
                val_preds.extend(probs.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_auc = roc_auc_score(val_labels, val_preds)

        fpr_v, tpr_v, thresholds_v = roc_curve(val_labels, val_preds)
        val_thresh = thresholds_v[np.argmax(tpr_v - fpr_v)]
        val_acc = accuracy_score(val_labels, np.array(val_preds) >= val_thresh)

        print(f"Epoch {epoch: 02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} (Acc: {val_acc:.4f}, Thresh: {val_thresh:.2f})")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            torch.save(model.state_dict(), best_model_path)
            print(f"   --> Saved BEST checkpoint for {display_name} (Epoch {epoch} | Val AUC: {best_val_auc:.4f})")
        scheduler.step()

    # Post-Training Evaluation
    print(f"\nLoading best checkpoint for {display_name} from Epoch {best_epoch}...")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()

    # Optimal threshold on validation set
    val_preds, val_labels = [], []
    with torch.no_grad():
        for signals, labels in val_loader:
            signals = signals.to(device)
            logits = model(signals).squeeze(-1)
            val_preds.extend(torch.sigmoid(logits).cpu().numpy())
            val_labels.extend(labels.numpy())

    fpr_v, tpr_v, thresholds_v = roc_curve(val_labels, val_preds)
    final_val_thresh = thresholds_v[np.argmax(tpr_v - fpr_v)]

    # Test evaluation
    test_preds, test_labels = [], []
    with torch.no_grad():
        for signals, labels in test_loader:
            signals = signals.to(device)
            logits = model(signals).squeeze(-1)
            test_preds.extend(torch.sigmoid(logits).cpu().numpy())
            test_labels.extend(labels.numpy())
            
    test_auc = roc_auc_score(test_labels, test_preds)
    test_acc = accuracy_score(test_labels, np.array(test_preds) >= final_val_thresh)

    print(f"Final Test AUC for {display_name}: {test_auc:.4f}")

    np.savez(
        best_outputs_path,
        y_test_true = np.array(test_labels),
        y_test_probs = np.array(test_preds),
        val_threshold = final_val_thresh,
        best_val_auc = best_val_auc,
        test_auc = test_auc, 
        test_acc = test_acc,
        epoch = best_epoch
    )

    all_summary_results.append({
        'Lead': display_name,
        'Val AUC': best_val_auc,
        'Test AUC': test_auc,
        'Test Acc': test_acc,
        'Val Threshold': final_val_thresh,
        'Best Epoch' : best_epoch
    })

# Step 6) Final summary table
# -------------------------------------------------------------------------
df_summary = pd.DataFrame(all_summary_results).sort_values(by='Test AUC', ascending=False).reset_index(drop = True)

print("\n" + "="*70)
print("FINAL CONSOLIDATED 15-EPOCH SINGLE-LEAD EVALUATION SUMMARY")
print("="*70)
print(df_summary.to_string(index = False, formatters={
    'Val AUC': '{:.4f}'.format,
    'Test AUC': '{:.4f}'.format,
    'Test Acc': '{:.4f}'.format,
    'Val Threshold': '{:.4f}'.format
}))
print("="*70)
